In [2]:
!pip install -U "huggingface_hub[cli]" hf_transfer evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 902.6 kB/s eta 0:00:001m828.9 kB/s eta 0:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.1.10
    Uninstalling hf-xet-1.1.10:
      Successfully uninstalled hf-xet-1.1.10
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.35.3
    Uninstalling huggingface-hub-0.35.3:
      Successfully uninstalled huggingface-hub-0.35.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tokenizers 0.21.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.1.4 which is incompatible.
transformers 4.52.4 requires huggingface-hub<1.0,>=0.30.0, but you have huggingface-hub 1.1.4 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
# ==================================================================================
# FIX CORRUPTED CONFIG & RUN FINAL EVALUATION
# ==================================================================================
import os
import shutil
import torch
import evaluate 
import numpy as np
from transformers import AutoTokenizer, AutoModel

# --- CONFIGURATION ---
LOCAL_ROBERTA_PATH = "/home/poorna/models/roberta-large"
# Ensure we have your predictions available. 
# If 'predictions' variable is lost, re-run the inference block first.
# Assuming 'predictions' and 'references' are still in memory from the previous cell.

# --- 1. FIX THE CORRUPTED MODEL ---
print(f"Checking integrity of {LOCAL_ROBERTA_PATH}...")

# Check for the specific symptom: Empty config.json
needs_download = False
config_path = os.path.join(LOCAL_ROBERTA_PATH, "config.json")

if os.path.exists(LOCAL_ROBERTA_PATH):
    if not os.path.exists(config_path) or os.path.getsize(config_path) == 0:
        print("⚠️ Found corrupted (0-byte) config files. Deleting folder...")
        shutil.rmtree(LOCAL_ROBERTA_PATH)
        needs_download = True
    else:
        print("✅ Model appears valid.")
else:
    needs_download = True

if needs_download:
    # CRITICAL FIX: Disable high-speed transfer to prevent 0-byte errors
    if "HF_HUB_ENABLE_HF_TRANSFER" in os.environ:
        del os.environ["HF_HUB_ENABLE_HF_TRANSFER"]
    
    print("⬇️ Downloading roberta-large (Standard Safe Mode)...")
    try:
        os.makedirs(LOCAL_ROBERTA_PATH, exist_ok=True)
        
        # Download
        tokenizer = AutoTokenizer.from_pretrained("roberta-large")
        model = AutoModel.from_pretrained("roberta-large")
        
        # Save
        tokenizer.save_pretrained(LOCAL_ROBERTA_PATH)
        model.save_pretrained(LOCAL_ROBERTA_PATH)
        
        # Verify
        new_size = os.path.getsize(os.path.join(LOCAL_ROBERTA_PATH, "config.json"))
        print(f"✅ Download complete. Config size: {new_size} bytes (Valid if > 0)")
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        exit()

# --- 2. RUN EVALUATION ---
print(f"\n--- Computing BERTScore with {LOCAL_ROBERTA_PATH} ---")
    # Check if we have predictions to evaluate
if 'predictions' not in globals() or 'references' not in globals():
    print("⚠️ No predictions found in memory. Please re-run the Inference Step above first!")
else:
    bertscore = evaluate.load("bertscore")
    
    # Compute with the guaranteed local model
    bert_results = bertscore.compute(
        predictions=predictions, 
        references=references, 
        model_type=LOCAL_ROBERTA_PATH,
        num_layers=17,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    
    print(f"\n=== BERTSCORE (roberta-large) ===")
    print(f"Precision: {np.mean(bert_results['precision']):.4f}")
    print(f"Recall:    {np.mean(bert_results['recall']):.4f}")
    print(f"F1:        {np.mean(bert_results['f1']):.4f}")

Checking integrity of /home/poorna/models/roberta-large...
✅ Model appears valid.

--- Computing BERTScore with /home/poorna/models/roberta-large ---

=== BERTSCORE (roberta-large) ===
Precision: 0.9087
Recall:    0.8951
F1:        0.9017
